In [0]:
#FASE 1 - EXPLORACIÓN Y DIAGNÓSTICO (Data Profiling)
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

In [0]:
import pandas as pd
import numpy as np

In [0]:
df = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")

In [0]:
df = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas").toPandas()
print("=== 1.1 PRIMERAS 10 FILAS ===")
print(df.head(10))
print("\n=== ÚLTIMAS 5 FILAS ===")
print(df.tail(5))
print(f"\nTotal de registros: {df.shape[0]} | Total de columnas: {df.shape[1]}")

In [0]:
# Identificación de tipos
print("\n=== 1.2 TIPOS DE DATOS ===")
print(df.dtypes)
df.info()

In [0]:
# Detección de nulos 
print("\n=== 1.3 NULOS POR COLUMNA ===")
nulos = pd.DataFrame({ "nulos": df.isnull().sum(),"porcentaje_%": (df.isnull().mean() * 100).round(2),})
print(nulos)
print(f"Total de nulos en la tabla: {int(df.isnull().sum().sum())}")
 

In [0]:
#Estadistica descriptiva
print("\n=== 1.4 ESTADÍSTICA DESCRIPTIVA (antes de limpiar) ===")
est = df[["cantidad", "precio_unitario"]].agg(["mean", "median", "std", "min", "max"])
print(est.round(2))
print("\n(describe() completo)")
print(df[["cantidad", "precio_unitario"]].describe().round(2))

In [0]:
#FASE 2 - LIMPIEZA Y DEPURACIÓN (Data Cleaning)
print("\n=== 2.1 DUPLICADOS ===")
print("Filas completamente duplicadas :", df.duplicated().sum())
print("id_transaccion repetidos       :", df.duplicated(subset=["id_transaccion"]).sum())
print(df[df.duplicated(subset=["id_transaccion"], keep=False)].sort_values("id_transaccion"))
 
filas_antes = len(df)
df = df.drop_duplicates(subset=["id_transaccion"], keep="first").reset_index(drop=True)
print(f"Filas antes: {filas_antes} -> después: {len(df)}  (eliminadas: {filas_antes - len(df)})")

In [0]:
#Normalización de texto 
print("\n=== 2.2 NORMALIZACIÓN DE metodo_pago ===")
print("Antes:\n", df["metodo_pago"].value_counts())
df["metodo_pago"] = df["metodo_pago"].replace({"Tarjeta Credito": "Tarjeta de Crédito"})
print("Después:\n", df["metodo_pago"].value_counts())

In [0]:
#Gestión de valores nulos 
print("\n=== 2.3 IMPUTACIÓN DE NULOS ===")
mediana_cant = df["cantidad"].median()
moda_satisf = df["satisfaccion_cliente"].mode()[0]
print(f"Mediana de cantidad            : {mediana_cant}")
print(f"Moda de satisfaccion_cliente   : {moda_satisf}")
 
df["cantidad"] = df["cantidad"].fillna(mediana_cant)
df["satisfaccion_cliente"] = df["satisfaccion_cliente"].fillna(moda_satisf)

In [0]:
#Tratamiento de atípicos (outlier 9999.00) 
print("\n=== 2.4 OUTLIER precio_unitario = 9999 ===")
print(df[df["precio_unitario"] == 9999.00])
 
#IMPUTAR con el precio medio de su categoría (Ropa), sin incluir el outlier, para conservar la transacción (cantidad, fecha, etc.).
mask_out = df["precio_unitario"] == 9999.00
media_cat = df.loc[~mask_out].groupby("categoria_producto")["precio_unitario"].mean()
print("\nPrecio medio por categoría (sin outlier):\n", media_cat.round(2))
 
df.loc[mask_out, "precio_unitario"] = (df.loc[mask_out, "categoria_producto"].map(media_cat))
print("\nFila corregida:")
print(df.loc[mask_out])
 
# Los precios nulos (6) también se imputan con el promedio de su categoría de lo contrario monto_total quedaría vacío 
media_cat2 = df.groupby("categoria_producto")["precio_unitario"].transform("mean")
n_prec_nulos = df["precio_unitario"].isnull().sum()
df["precio_unitario"] = df["precio_unitario"].fillna(media_cat2)
print(f"\nPrecios nulos imputados por media de categoría: {n_prec_nulos}")
 
# Los precios imputados se redondean a 2 decimales (formato de moneda)
df["precio_unitario"] = df["precio_unitario"].round(2)
 
# Con los nulos resueltos, ahora sí se pueden convertir a entero
df["cantidad"] = df["cantidad"].astype(int)
df["satisfaccion_cliente"] = df["satisfaccion_cliente"].astype(int)
 
print("\nNulos restantes:\n", df.isnull().sum())
print("\nEstadística descriptiva DESPUÉS de limpiar:")
print(df[["cantidad", "precio_unitario"]].agg(["mean", "median", "std", "min", "max"]).round(2))

In [0]:
# FASE 3 TRANSFORMACIÓN E INGENIERÍA DE ATRIBUTOS
df["monto_total"] = (df["cantidad"] * df["precio_unitario"]).round(2)
print("\n=== 3.1 monto_total ===")
print(df[["id_transaccion", "cantidad", "precio_unitario", "monto_total"]].head())

In [0]:
#Conversión temporal 
df["fecha"] = pd.to_datetime(df["fecha"])
df["dia_semana"] = df["fecha"].dt.day_name()
df["mes"] = df["fecha"].dt.month
print("\n=== 3.2 FECHA ===")
print(df.dtypes[["fecha"]])
print(df[["fecha", "dia_semana", "mes"]].head())

In [0]:
#Discretización (binning) 
# Bajo: < 100 | Medio: 100 a 500 (inclusive) | Alto: > 500
import numpy as np
 
def rango_monto(m):
    if m < 100: return "Bajo" 
    elif m <= 500:  return "Medio" 
    return "Alto"
 
df["rango_monto"] = df["monto_total"].apply(rango_monto)
# Equivalente con pd.cut (de la caja de herramientas):
# bins = [-np.inf, 99.9999, 500, np.inf]  -> (right=True)
rango_cut = pd.cut(df["monto_total"], bins=[-np.inf, 99.9999, 500, np.inf],labels=["Bajo", "Medio", "Alto"])
assert (rango_cut.astype(str) == df["rango_monto"]).all()
print("\n=== 3.3 RANGOS DE MONTO ===")
print(df["rango_monto"].value_counts().reindex(["Bajo", "Medio", "Alto"]))

In [0]:
# FASE AGREGACIÓN Y RESUMEN DE DATOS
#Group By 
print("\n=== 4.1 INGRESOS Y UNIDADES POR CATEGORÍA ===")
res_cat = ( df.groupby("categoria_producto").agg(ingresos_totales=("monto_total", "sum"), promedio_unidades=("cantidad", "mean")).round(2).sort_values("ingresos_totales", ascending=False))
print(res_cat)
 

In [0]:
# Tabla dinámica 
print("\n=== 4.2 PIVOT: monto_total por región x categoría ===")
pivot = pd.pivot_table(df, values="monto_total", index="region", columns="categoria_producto", aggfunc="sum", fill_value=0, margins=True, margins_name="Total").round(2)
print(pivot)
 

In [0]:
# Análisis de satisfacción
print("\n=== 4.3 SATISFACCIÓN MEDIA POR MÉTODO DE PAGO ===")
satisf = (df.groupby("metodo_pago")["satisfaccion_cliente"].mean().round(2).sort_values(ascending=False))
print(satisf)
 

In [0]:
# FASE 5 - SELECCIÓN, FILTRADO Y MUESTREO
# Filtrado multicriterio 
print("\n=== 5.1 NORTE y SATISFACCIÓN >= 4 ===")
filtro = df[(df["region"] == "Norte") & (df["satisfaccion_cliente"] >= 4)]
print(f"Registros que cumplen: {len(filtro)}")
print(filtro)

In [0]:
#Muestreo estratificado
print("\n=== 5.2 MUESTRA ESTRATIFICADA 10% POR REGIÓN ===")
muestra = df.groupby("region").sample(frac=0.10, random_state=42)
print(f"Tamaño de la muestra: {len(muestra)} de {len(df)}")
print("\nProporción por región (población vs muestra):")
comp = pd.DataFrame({"poblacion_%": (df["region"].value_counts(normalize=True) * 100).round(1), "muestra_%": (muestra["region"].value_counts(normalize=True) * 100).round(1), "n_muestra": muestra["region"].value_counts(),})
print(comp)
print(muestra)

In [0]:
# EXPORTAR RESULTADOS
df.to_csv("ventas_limpio.csv", index=False, encoding="utf-8-sig")
print("\nArchivo limpio guardado: ventas_limpio.csv")